# VLM³ — Metric Depth Estimation Demo

This notebook runs the released `facebook/VLM3-depth` model on the sample image
`sample_data/depth.jpeg` and 100 ground-truth points stored in
`sample_data/depth_labels.json`, then reports the `DepthLM`-style accuracy of
the predictions.

## What the sample data contains
- `depth.jpeg` — an image already preprocessed by the eval pipeline:
  undistorted (when fx ≠ fy) and resized so that the focal length is 1000 px.
- `depth_labels.json` — for each of 100 labelled points:
  - `pixel_coord_resized` — pixel `(x, y)` in the resized image
  - `pixel_coord_normalized_2000` — same point rescaled to `[0, 2000]`
  - `depth_meters` — ground-truth distance from the camera (in metres)

## What the prompt looks like
```
Given this image, how far is the point at coordinates (norm_x, norm_y)
from the camera? The coordinates are in normalised [0, 2000] format relative
to image width and height. Output the thinking process in <think> </think>
and final answer (the meter number only, without the unit) in
<answer> </answer> tags.
```

## Accuracy metric (matches `DepthLM`)
$\delta_1$ accuracy: a prediction is *correct* when
$\max(\hat d / d_{\text{gt}},\; d_{\text{gt}} / \hat d) < 1.25$,
where $\hat d$ is the predicted depth and $d_{\text{gt}}$ is the ground truth.

In [ ]:
import json
import os
import re
from pathlib import Path

import torch
from PIL import Image
from tqdm import tqdm
from transformers import AutoModelForImageTextToText, AutoProcessor

# ---- Paths ----
REPO_ROOT = Path.cwd()
IMAGE_PATH = REPO_ROOT / "sample_data" / "depth.jpeg"
LABELS_PATH = REPO_ROOT / "sample_data" / "depth_labels.json"

assert IMAGE_PATH.exists(), f"Missing image: {IMAGE_PATH}"
assert LABELS_PATH.exists(), f"Missing labels: {LABELS_PATH}"

with open(LABELS_PATH) as f:
    labels = json.load(f)

print(f"Image:        {IMAGE_PATH}")
print(f"Image size:   {labels['image_size']}")
print(f"#GT points:   {labels['num_points']}")
print(f"Coord system: {labels['coordinate_system']}")

In [ ]:
# ---- Load VLM3 once ----
MODEL_ID = "facebook/VLM3-depth"

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, dtype="auto", device_map="auto"
)
processor = AutoProcessor.from_pretrained(MODEL_ID)
model.eval()
print("Model loaded.")

In [ ]:
# Load the local image once and reuse it for every query
image = Image.open(IMAGE_PATH).convert("RGB")

PROMPT_TEMPLATE = (
    "Given this image, how far is the point at coordinates ({nx}, {ny}) "
    "from the camera? The coordinates are in normalised [0, 2000] format relative "
    "to image width and height. Output the thinking process in <think> </think> "
    "and final answer (the meter number only, without the unit) in <answer> </answer> tags."
)

_ANSWER_RE = re.compile(r"<answer>\s*([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)\s*</answer>")


def parse_depth(text):
    """Extract the float inside <answer>...</answer>; return None if missing/invalid."""
    m = _ANSWER_RE.search(text)
    if not m:
        return None
    try:
        return float(m.group(1))
    except ValueError:
        return None


@torch.inference_mode()
def predict_depth(norm_x, norm_y, max_new_tokens=256):
    """Run VLM3 on (norm_x, norm_y) and return (parsed_depth_meters, raw_text)."""
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": PROMPT_TEMPLATE.format(nx=norm_x, ny=norm_y)},
            ],
        }
    ]
    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    generated_ids = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    trimmed = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    text = processor.batch_decode(
        trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )[0]
    return parse_depth(text), text

In [ ]:
# ---- Sanity check on the first sample ----
first = labels["samples"][0]
nx, ny = first["pixel_coord_normalized_2000"]
gt = first["depth_meters"]

pred, raw = predict_depth(nx, ny)
print(f"Coord (normalised [0,2000]): ({nx}, {ny})")
print(f"Ground truth depth          : {gt:.3f} m")
print(f"Predicted depth             : {pred} m")
print("--- raw model output ---")
print(raw)

In [ ]:
# ---- Run inference over all 100 GT points ----
# Set MAX_POINTS to e.g. 25 for a quick demo; keep None to evaluate all 100.
MAX_POINTS = None

samples = labels["samples"]
if MAX_POINTS is not None:
    samples = samples[:MAX_POINTS]

results = []
for s in tqdm(samples, desc="VLM3 depth eval"):
    nx, ny = s["pixel_coord_normalized_2000"]
    gt = s["depth_meters"]
    pred, raw = predict_depth(nx, ny)
    results.append({
        "norm_xy": (nx, ny),
        "gt": gt,
        "pred": pred,
        "raw": raw,
    })

print(f"Done: {len(results)} predictions.")

In [ ]:
# ---- Compute DepthLM delta_1 accuracy ----
# A prediction is correct when max(pred/gt, gt/pred) < 1.25.
DELTA = 1.25

valid = [(r["pred"], r["gt"]) for r in results if r["pred"] is not None and r["pred"] > 0]
parse_failures = sum(1 for r in results if r["pred"] is None)

n_correct = sum(
    1 for pred, gt in valid if max(pred / gt, gt / pred) < DELTA
)
n = len(valid)

print(f"Total samples     : {len(results)}")
print(f"Parse failures    : {parse_failures}")
print(f"Valid predictions : {n}")
if n > 0:
    print()
    print(f"delta_1 accuracy (delta < {DELTA}): {n_correct}/{n}  ({100.0 * n_correct / n:5.1f}%)")

In [ ]:
# ---- Per-sample table (first 15) ----
import pandas as pd

df = pd.DataFrame([
    {
        "norm_x": r["norm_xy"][0],
        "norm_y": r["norm_xy"][1],
        "gt_m": round(r["gt"], 3),
        "pred_m": (None if r["pred"] is None else round(r["pred"], 3)),
        "abs_err_m": (None if r["pred"] is None else round(abs(r["pred"] - r["gt"]), 3)),
        "ratio": (
            None if r["pred"] is None or r["pred"] <= 0
            else round(max(r["pred"] / r["gt"], r["gt"] / r["pred"]), 3)
        ),
        "delta1_ok": (
            None if r["pred"] is None or r["pred"] <= 0
            else max(r["pred"] / r["gt"], r["gt"] / r["pred"]) < 1.25
        ),
    }
    for r in results
])
df.head(15)

---

# VLM³ — Object-level 3D Understanding Demo (`facebook/VLM3-object`)

This section runs the released `facebook/VLM3-object` model on the SpatialRGPT-Bench sample we saved at `sample_data/spatialrgpt.jpeg` + `sample_data/spatialrgpt_labels.json`, and compares the prediction with the ground-truth answer.

For SpatialRGPT-style questions the prompt references objects via their **bounding boxes in `[0, 1000]` normalized coordinates**.

Example prompt:
```
Can you confirm if bounding box region (0, 39, 652, 999) is smaller than
bounding box region (6, 380, 228, 562)?
```

In [ ]:
# ---- Load SpatialRGPT sample ----
SRGPT_IMAGE_PATH = REPO_ROOT / "sample_data" / "spatialrgpt.jpeg"
SRGPT_LABELS_PATH = REPO_ROOT / "sample_data" / "spatialrgpt_labels.json"

assert SRGPT_IMAGE_PATH.exists(), f"Missing image: {SRGPT_IMAGE_PATH}"
assert SRGPT_LABELS_PATH.exists(), f"Missing labels: {SRGPT_LABELS_PATH}"

with open(SRGPT_LABELS_PATH) as f:
    srgpt = json.load(f)

srgpt_image = Image.open(SRGPT_IMAGE_PATH).convert("RGB")

print(f"Image:        {SRGPT_IMAGE_PATH}")
print(f"Image size:   {srgpt['image_size_resized']}")
print(f"# bboxes:     {len(srgpt['bboxes'])}")
print(f"QA category:  {srgpt['qa_info'].get('category')}")
print(f"QA type:      {srgpt['qa_info'].get('type')}")
print()
print("Prompt:")
print(f"  {srgpt['prompt']}")
print()
print("Ground-truth answer:")
print(f"  {srgpt['answer']}")

In [ ]:
# ---- Optional: visualise the bounding boxes on the resized image ----
# Uses the `bbox_resized_pixels` coords (which match the saved JPEG) so
# the overlay aligns perfectly with the image.
from PIL import ImageDraw

vis = srgpt_image.copy()
draw = ImageDraw.Draw(vis)
colors = ["red", "limegreen", "deepskyblue", "orange", "magenta"]
for i, b in enumerate(srgpt["bboxes"]):
    x1, y1, x2, y2 = b["bbox_resized_pixels"]
    color = colors[i % len(colors)]
    draw.rectangle([x1, y1, x2, y2], outline=color, width=4)
    draw.text((x1 + 4, y1 + 4), f"Region {b['region_id']}", fill=color)
vis

In [ ]:
# ---- Load VLM3-object (separate checkpoint from VLM3-depth) ----
# Free the depth model first if it's still in memory.
try:
    del model, processor
except NameError:
    pass
import gc
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

OBJECT_MODEL_ID = "facebook/VLM3-object"

object_model = AutoModelForImageTextToText.from_pretrained(
    OBJECT_MODEL_ID, dtype="auto", device_map="auto"
)
object_processor = AutoProcessor.from_pretrained(OBJECT_MODEL_ID)
object_model.eval()
print(f"Loaded {OBJECT_MODEL_ID}.")

In [ ]:
# ---- Run inference on the SpatialRGPT sample ----
@torch.inference_mode()
def run_object_inference(image, prompt, max_new_tokens=512):
    """Run VLM3-object on (image, text) and return the generated text."""
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt},
            ],
        }
    ]
    inputs = object_processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    ).to(object_model.device)

    generated_ids = object_model.generate(
        **inputs, max_new_tokens=max_new_tokens, do_sample=False
    )
    trimmed = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    return object_processor.batch_decode(
        trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )[0]


prediction = run_object_inference(srgpt_image, srgpt["prompt"])

print("=" * 70)
print("Prompt:")
print("  " + srgpt["prompt"])
print()
print("Ground-truth answer:")
print("  " + srgpt["answer"])
print()
print("VLM3-object prediction:")
print("  " + prediction)
print("=" * 70)

In [ ]:
# ---- Lightweight binary correctness check for SpatialRGPT qualitative QAs ----
# SpatialRGPT-Bench qualitative questions expect either an affirmation or
# negation. We look for the leading polarity token ("correct"/"incorrect",
# "yes"/"no") and compare with the GT.
import re as _re

_POSITIVE = _re.compile(r"\b(yes|correct|affirmative|true|right|indeed)\b", _re.IGNORECASE)
_NEGATIVE = _re.compile(r"\b(no|incorrect|wrong|false|not)\b", _re.IGNORECASE)


def polarity(text):
    pos = _POSITIVE.search(text)
    neg = _NEGATIVE.search(text)
    if pos and not neg:
        return "positive"
    if neg and not pos:
        return "negative"
    if pos and neg:
        return "positive" if pos.start() < neg.start() else "negative"
    return "unknown"


gt_polarity = polarity(srgpt["answer"])
pred_polarity = polarity(prediction)
match = gt_polarity == pred_polarity and gt_polarity != "unknown"

print(f"GT polarity       : {gt_polarity}")
print(f"Predicted polarity: {pred_polarity}")
print(f"Match             : {match}")

---

# VLM³ — Pixel Correspondence Demo (`facebook/VLM3-corr`)

This section runs the released `facebook/VLM3-corr` model on the UFM ETH3D pixel-correspondence sample we saved at `sample_data/pixel_corr_img1.jpeg` + `sample_data/pixel_corr_img2.jpeg` + `sample_data/pixel_corr_labels.json`, and compares the prediction with the ground-truth correspondence.

## Preprocessing
Each image is resized so that `max(width, height) == 1024`. Which makes the inference lighter than normalizing the focal length. Pixel coordinates in the prompt are normalised to `[0, 1000]` relative to the resized image width/height.

## Prompt template
```
Given these two images, what pixel in the second image corresponds to pixel
(x1, y1) in the first image? Report the answer as (x, y).
```

## Accuracy metric
End-Point Error (EPE) in `[0, 1000]` normalised pixels (lower is better).

In [ ]:
# ---- Load pixel-correspondence sample ----
PC_IMG1_PATH = REPO_ROOT / "sample_data" / "pixel_corr_img1.jpeg"
PC_IMG2_PATH = REPO_ROOT / "sample_data" / "pixel_corr_img2.jpeg"
PC_LABELS_PATH = REPO_ROOT / "sample_data" / "pixel_corr_labels.json"

for p in (PC_IMG1_PATH, PC_IMG2_PATH, PC_LABELS_PATH):
    assert p.exists(), f"Missing: {p}"

with open(PC_LABELS_PATH) as f:
    pc = json.load(f)

pc_img1 = Image.open(PC_IMG1_PATH).convert("RGB")
pc_img2 = Image.open(PC_IMG2_PATH).convert("RGB")

print(f"Image 1: {PC_IMG1_PATH}  ({pc['image_1_size_resized']})")
print(f"Image 2: {PC_IMG2_PATH}  ({pc['image_2_size_resized']})")
print(f"Scene:   {pc['scene_name']} ({pc['dataset_name']})")
print(f"# corrs: {pc['num_correspondences']}")
print()
print("Example prompt:")
print("  " + pc["example_prompt"])
print("Example GT (norm-1000):", pc["example_gt_norm_1000"])

In [ ]:
# ---- Optional: visualise the first 8 correspondences side by side ----
from PIL import ImageDraw

vis1 = pc_img1.copy()
vis2 = pc_img2.copy()
d1 = ImageDraw.Draw(vis1)
d2 = ImageDraw.Draw(vis2)
palette = [
    "red", "limegreen", "deepskyblue", "orange", "magenta",
    "yellow", "cyan", "hotpink",
]
for i, c in enumerate(pc["correspondences"][:8]):
    color = palette[i % len(palette)]
    x1, y1 = c["img1_pixel_in_resized"]
    x2, y2 = c["img2_pixel_in_resized"]
    r = 6
    d1.ellipse([x1 - r, y1 - r, x1 + r, y1 + r], outline=color, width=3)
    d2.ellipse([x2 - r, y2 - r, x2 + r, y2 + r], outline=color, width=3)
    d1.text((x1 + 8, y1 - 4), str(i), fill=color)
    d2.text((x2 + 8, y2 - 4), str(i), fill=color)

# Concat for display
from PIL import Image as _PILImage
h = max(vis1.height, vis2.height)
concat = _PILImage.new("RGB", (vis1.width + vis2.width + 10, h), "black")
concat.paste(vis1, (0, 0))
concat.paste(vis2, (vis1.width + 10, 0))
concat

In [ ]:
# ---- Load VLM3-correspondence (separate checkpoint) ----
# Free the previously-loaded models if present.
for var_name in ("model", "processor", "object_model", "object_processor"):
    try:
        del globals()[var_name]
    except KeyError:
        pass
import gc
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

PC_MODEL_ID = "facebook/VLM3-corr"

pc_model = AutoModelForImageTextToText.from_pretrained(
    PC_MODEL_ID, dtype="auto", device_map="auto"
)
pc_processor = AutoProcessor.from_pretrained(PC_MODEL_ID)
pc_model.eval()
print(f"Loaded {PC_MODEL_ID}.")

In [ ]:
# ---- Helpers: prompt formatter, output parser, and batched runner ----
PC_PROMPT_TEMPLATE = pc["prompt_template"]
PC_THINK_INSTR = pc["thinking_format_instruction"]
USE_THINK = True   # set False to disable <think> reasoning

_PRED_RE = re.compile(
    r"\(?\s*(-?\d+(?:\.\d+)?)\s*,\s*(-?\d+(?:\.\d+)?)\s*\)?"
)


def parse_xy(text):
    """Extract the (x, y) prediction from the model output."""
    cleaned = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()
    m = _PRED_RE.search(cleaned)
    if not m:
        return None, None
    return float(m.group(1)), float(m.group(2))


@torch.inference_mode()
def run_pixel_corr_inference(img1, img2, x1_norm, y1_norm, max_new_tokens=2048 if USE_THINK else 256):
    prompt = PC_PROMPT_TEMPLATE.format(x1=int(round(x1_norm)), y1=int(round(y1_norm)))
    if USE_THINK:
        prompt = prompt + PC_THINK_INSTR
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": img1},
                {"type": "image", "image": img2},
                {"type": "text",  "text": prompt},
            ],
        }
    ]
    inputs = pc_processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    ).to(pc_model.device)

    generated_ids = pc_model.generate(
        **inputs, max_new_tokens=max_new_tokens, do_sample=False
    )
    trimmed = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    text = pc_processor.batch_decode(
        trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )[0]
    px, py = parse_xy(text)
    return (px, py), text

In [ ]:
# ---- Sanity check on the first correspondence ----
first = pc["correspondences"][0]
x1n, y1n = first["img1_norm_1000"]
gt2 = first["img2_norm_1000"]
(pred_x, pred_y), raw = run_pixel_corr_inference(pc_img1, pc_img2, x1n, y1n)
print("=" * 70)
print(f"Query (img1, norm-1000): ({x1n:.1f}, {y1n:.1f})")
print(f"GT    (img2, norm-1000): ({gt2[0]:.1f}, {gt2[1]:.1f})")
print(f"Pred  (img2, norm-1000): ({pred_x}, {pred_y})")
if pred_x is not None:
    epe = math.hypot(pred_x - gt2[0], pred_y - gt2[1])
    print(f"EPE                      : {epe:.2f} norm-1000 pixels")
print("--- raw model output ---")
print(raw)
print("=" * 70)

In [ ]:
# ---- Evaluate over all correspondences in the sample ----
MAX_CORRS = None   # or e.g. 16 for a quick smoke test

corrs = pc["correspondences"]
if MAX_CORRS is not None:
    corrs = corrs[:MAX_CORRS]

results_pc = []
for c in tqdm(corrs, desc="VLM3-correspondence eval"):
    x1n, y1n = c["img1_norm_1000"]
    gt2 = c["img2_norm_1000"]
    (pred_x, pred_y), raw = run_pixel_corr_inference(pc_img1, pc_img2, x1n, y1n)
    epe = (
        math.hypot(pred_x - gt2[0], pred_y - gt2[1])
        if pred_x is not None else None
    )
    results_pc.append({
        "query": (x1n, y1n),
        "gt":    tuple(gt2),
        "pred":  (pred_x, pred_y),
        "epe":   epe,
        "raw":   raw,
    })

print(f"Done: {len(results_pc)} predictions.")

In [ ]:
# ---- Compute End-Point Error (EPE) ----
valid_epes = [r["epe"] for r in results_pc if r["epe"] is not None]
n_total = len(results_pc)
n_parsed = len(valid_epes)
n_failed = n_total - n_parsed

print(f"Total queries     : {n_total}")
print(f"Parse failures    : {n_failed}")
print(f"Valid predictions : {n_parsed}")
if n_parsed > 0:
    epes = valid_epes
    mean_epe = sum(epes) / len(epes)
    median_epe = sorted(epes)[len(epes) // 2]
    print()
    print("--- EPE (lower is better, in norm-1000 px) ---")
    print(f"  mean EPE   : {mean_epe:.3f}")
    print(f"  median EPE : {median_epe:.3f}")

# Per-sample table for quick inspection
import pandas as pd
pd.DataFrame([
    {
        "q_x":  round(r["query"][0], 1),
        "q_y":  round(r["query"][1], 1),
        "gt_x": round(r["gt"][0], 1),
        "gt_y": round(r["gt"][1], 1),
        "pred_x": (None if r["pred"][0] is None else round(r["pred"][0], 1)),
        "pred_y": (None if r["pred"][1] is None else round(r["pred"][1], 1)),
        "EPE":   (None if r["epe"] is None else round(r["epe"], 2)),
    }
    for r in results_pc
]).head(15)

---

# VLM³ — Camera Pose Estimation Demo (`facebook/VLM3-pose`)

This section runs the released `facebook/VLM3-pose` model on a ScanNet++v2 image pair we saved at `sample_data/pose_img1.jpeg` + `sample_data/pose_img2.jpeg` + `sample_data/pose_labels.json`, and compares the predictions with the ground-truth pose values.

## Preprocessing 
Both images are resized so that **focal length = 750 px**.

## Conventions
- **Rotation** is described as intrinsic **Yaw → Pitch → Roll** about cam-1's body axes:
  - yaw > 0 → turn right
  - pitch > 0 → look up
  - roll > 0 → bank right (image rotates clockwise)
- **Translation** is a unit vector in cam-1's body frame: `x > 0` right, `y > 0` down, `z > 0` forward.
- **Distance** is in metres (only emitted for `metric` / `estimated metric` datasets).

## Ground-truth answer formats (used to parse model output)
```
Rotation angle: 89.7 degrees
Yaw=+89.6°, Pitch=+2.6°, Roll=+6.1°
The camera moves left, forward, unit vector (-0.69, +0.02, +0.72).
Translation distance: 1.98 meters
```

In [ ]:
# ---- Load camera-pose sample ----
POSE_IMG1_PATH = REPO_ROOT / "sample_data" / "pose_img1.jpeg"
POSE_IMG2_PATH = REPO_ROOT / "sample_data" / "pose_img2.jpeg"
POSE_LABELS_PATH = REPO_ROOT / "sample_data" / "pose_labels.json"

for p in (POSE_IMG1_PATH, POSE_IMG2_PATH, POSE_LABELS_PATH):
    assert p.exists(), f"Missing: {p}"

with open(POSE_LABELS_PATH) as f:
    pose = json.load(f)

pose_img1 = Image.open(POSE_IMG1_PATH).convert("RGB")
pose_img2 = Image.open(POSE_IMG2_PATH).convert("RGB")

print(f"Image 1: {POSE_IMG1_PATH}  ({pose['image_1_size_resized']})")
print(f"Image 2: {POSE_IMG2_PATH}  ({pose['image_2_size_resized']})")
print(f"Dataset: {pose['dataset_name']}  ({pose['scale_type']})")
print(f"# QA pairs: {len(pose['qa_pairs'])}")
print()
print("Ground-truth pose:")
for k, v in pose["ground_truth_pose"].items():
    print(f"  {k:<26s}: {v}")

In [ ]:
# ---- Show the two input images side by side ----
from PIL import Image as _PILImage

def hstack(im1, im2, gap=10):
    h = max(im1.height, im2.height)
    out = _PILImage.new("RGB", (im1.width + im2.width + gap, h), "black")
    out.paste(im1, (0, 0))
    out.paste(im2, (im1.width + gap, 0))
    # Display thumbnail to fit the notebook viewport
    out.thumbnail((1600, 800))
    return out

hstack(pose_img1, pose_img2)

In [ ]:
# ---- Load VLM3-pose (separate checkpoint) ----
for var_name in (
    "model", "processor",
    "object_model", "object_processor",
    "pc_model", "pc_processor",
):
    try:
        del globals()[var_name]
    except KeyError:
        pass
import gc
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

POSE_MODEL_ID = "facebook/VLM3-pose"

pose_model = AutoModelForImageTextToText.from_pretrained(
    POSE_MODEL_ID, dtype="auto", device_map="auto"
)
pose_processor = AutoProcessor.from_pretrained(POSE_MODEL_ID)
pose_model.eval()
print(f"Loaded {POSE_MODEL_ID}.")

In [ ]:
# ---- Helpers: parsers + single-question inference ----
_ANGLE_RE = re.compile(r"([-+]?\d+(?:\.\d+)?)\s*(?:degrees|deg|\u00b0)", re.IGNORECASE)
_YAW_RE   = re.compile(r"yaw\s*=?\s*([-+]?\d+(?:\.\d+)?)", re.IGNORECASE)
_PITCH_RE = re.compile(r"pitch\s*=?\s*([-+]?\d+(?:\.\d+)?)", re.IGNORECASE)
_ROLL_RE  = re.compile(r"roll\s*=?\s*([-+]?\d+(?:\.\d+)?)", re.IGNORECASE)
_VEC_RE   = re.compile(
    r"\(?\s*([-+]?\d+(?:\.\d+)?)\s*,\s*"
    r"([-+]?\d+(?:\.\d+)?)\s*,\s*"
    r"([-+]?\d+(?:\.\d+)?)\s*\)?"
)
_DIST_RE  = re.compile(r"([-+]?\d+(?:\.\d+)?)\s*(?:meters|m)\b", re.IGNORECASE)


def parse_pose_output(qid: str, text: str):
    """Parse the model output for a given question id."""
    cleaned = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()
    if qid == "rotation_angle":
        m = _ANGLE_RE.search(cleaned)
        return {"angle_deg": float(m.group(1))} if m else {}
    if qid == "euler_natural":
        out = {}
        for key, regex in [("yaw", _YAW_RE), ("pitch", _PITCH_RE), ("roll", _ROLL_RE)]:
            m = regex.search(cleaned)
            if m:
                out[f"{key}_deg"] = float(m.group(1))
        return out
    if qid == "translation_direction":
        m = _VEC_RE.search(cleaned)
        return {"unit_vector": [float(m.group(1)), float(m.group(2)), float(m.group(3))]} if m else {}
    if qid == "translation_distance":
        m = _DIST_RE.search(cleaned)
        return {"distance_m": float(m.group(1))} if m else {}
    return {}


@torch.inference_mode()
def run_pose_inference(img1, img2, question, max_new_tokens=512):
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": img1},
                {"type": "image", "image": img2},
                {"type": "text",  "text": question},
            ],
        }
    ]
    inputs = pose_processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    ).to(pose_model.device)

    generated_ids = pose_model.generate(
        **inputs, max_new_tokens=max_new_tokens, do_sample=False
    )
    trimmed = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    return pose_processor.batch_decode(
        trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )[0]

In [ ]:
# ---- Run the model on every QA pair, then compare to GT ----
import math

results_pose = []
for qa in tqdm(pose["qa_pairs"], desc="VLM3-pose eval"):
    raw = run_pose_inference(pose_img1, pose_img2, qa["question"])
    parsed = parse_pose_output(qa["question_id"], raw)
    results_pose.append({
        "qid":    qa["question_id"],
        "question": qa["question"],
        "gt":     qa["answer"],
        "pred":   raw,
        "parsed": parsed,
    })

for r in results_pose:
    print("=" * 70)
    print(f"[{r['qid']}]")
    print("  GT  :", r["gt"])
    print("  Pred:", r["pred"].strip())
    print("  Parsed:", r["parsed"])
print("=" * 70)

In [ ]:
# ---- Compute simple per-quantity errors ----
import math

gt = pose["ground_truth_pose"]
errors = {}

for r in results_pose:
    p = r["parsed"]
    if r["qid"] == "rotation_angle" and "angle_deg" in p:
        errors["|Δ rotation angle| (deg)"] = abs(p["angle_deg"] - gt["rotation_angle_deg"])
    elif r["qid"] == "euler_natural":
        for k in ("yaw", "pitch", "roll"):
            key = f"{k}_deg"
            if key in p:
                errors[f"|Δ {k}| (deg)"] = abs(p[key] - gt[key])
    elif r["qid"] == "translation_direction" and "unit_vector" in p:
        gt_vec_str = gt["translation_unit_vector"]
        # Parse '(x, y, z)' string from GT
        import ast
        try:
            gv = list(ast.literal_eval(gt_vec_str.replace("+", "")))
        except Exception:
            m = _VEC_RE.search(gt_vec_str)
            gv = [float(m.group(1)), float(m.group(2)), float(m.group(3))] if m else None
        if gv is not None:
            pv = p["unit_vector"]
            # Angular error between unit vectors
            dot = sum(a * b for a, b in zip(pv, gv))
            np_p = math.sqrt(sum(a * a for a in pv))
            np_g = math.sqrt(sum(b * b for b in gv))
            cos = max(-1.0, min(1.0, dot / (np_p * np_g + 1e-9)))
            errors["trans-direction angular error (deg)"] = math.degrees(math.acos(cos))
    elif r["qid"] == "translation_distance" and "distance_m" in p:
        errors["|Δ distance| (m)"]  = abs(p["distance_m"] - gt["translation_distance_m"])
        errors["|Δ distance| (rel)"] = (
            abs(p["distance_m"] - gt["translation_distance_m"])
            / max(abs(gt["translation_distance_m"]), 1e-6)
        )

print("--- Per-quantity errors (lower is better) ---")
for k, v in errors.items():
    print(f"  {k:<40s}: {v:.3f}")